# Beyond Attribute Means: Joint First- and Second-Order Change Detection in Popular Music

Reproducibility notebook for the LAMIR paper by Marco Antônio Machado and Flavio Figueiredo.

This notebook contains the analyses used in the manuscript: the annual joint representation,
the original-style mean-only baseline, centroid-displacement decomposition, 5,000 bootstrap
repetitions, 5,000 refit-scale relational-null realizations, seven category ablations,
Table 1, and Figures 1–3. Run the cells **in order from a fresh kernel**.

## Setup

Install `requirements.txt` and place `mgphot_genes.tsv` and `mgphot_gene_values.tsv` in `data/`.
The dataset is supplied separately; see `data/README.md`. Paths are relative to the repository root.
Full resampling is computationally expensive.
Set `QUICK_CHECK = True` only to check execution with 10 repetitions; those outputs are explicitly
separated from paper outputs and must not be reported as the paper's validation results.

The values in `reference/published_validation.csv` are used to compare computed validation results
with the paper. They are not inputs to the statistical calculations.

## 1. Configuration

In [ ]:
from pathlib import Path
import ast
import hashlib
import json
import platform
import importlib.metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.covariance import LedoitWolf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from scipy.signal import find_peaks
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits

# Limit BLAS threads: the repeated covariance problems are small.
_thread_limit = threadpool_limits(limits=1)
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
QUICK_CHECK = False
BOOTSTRAP_REPS = 10 if QUICK_CHECK else 5000
NULL_REFIT_REPS = 10 if QUICK_CHECK else 5000
BOOTSTRAP_SEED = 20260812
NULL_SEED = 20260813
BOOTSTRAP_TOP_K = 5
BOOTSTRAP_MATCH_TOLERANCE = 1
HALF_WINDOW = 5
LAMBDA_SECOND = 1.0
MIN_PEAK_DISTANCE = 4

OUTPUT_DIR = ROOT / ("quick_check" if QUICK_CHECK else "results")
FIGURE_DIR = OUTPUT_DIR / "figs" if QUICK_CHECK else ROOT / "figs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
GENE_METADATA_PATH = DATA_DIR / "mgphot_genes.tsv"
GENE_VALUES_PATH = DATA_DIR / "mgphot_gene_values.tsv"
for path in (GENE_METADATA_PATH, GENE_VALUES_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. See data/README.md or edit DATA_DIR.")

print("Mode:", "QUICK CHECK — not paper validation" if QUICK_CHECK else "PAPER (5,000 repetitions per test)")

## 2. Data and annual representation

MGPHot provides 58 attributes in [0,1]. Tracks are assigned to their first chart year and receive
equal weight. The paper uses 21,299 tracks across 1958–2022. Chart position and duration are not modeled.

Within each year we compute attribute means and Ledoit–Wolf regularized correlations.
Shrinkage stabilizes annual covariance estimation. We retain the 1,653 off-diagonal upper-triangle
entries, standardize each coordinate across years, and divide each block by the square root of
its dimension. Thus squared Euclidean distance adds the **average squared standardized change**
from each block. Correlations are aggregate associations, not causal or within-genre effects.

In [ ]:
# ============================================================
# 2. Load and inspect the data
# ============================================================

genes = pd.read_csv(GENE_METADATA_PATH, sep="\t")
tracks = pd.read_csv(GENE_VALUES_PATH, sep="\t")

gene_names = genes["name"].tolist()
gene_categories = genes["category"].tolist()

track_vectors = tracks["gene_values"].apply(ast.literal_eval)
X_tracks = np.vstack(track_vectors.to_numpy()).astype(float)
track_years = tracks["year"].to_numpy(dtype=int)
years = np.array(sorted(tracks["year"].unique()))

if X_tracks.shape[1] != len(gene_names):
    raise ValueError("The number of gene columns does not match the metadata.")

tracks_per_year = tracks.groupby("year").size().rename("n_tracks")

print("Track × attribute matrix:", X_tracks.shape)
print("Number of attributes:", len(gene_names))
print("Years:", years.min(), "to", years.max())
print("Number of years:", len(years))
print("Tracks per year: min =", tracks_per_year.min(),
      ", median =", int(tracks_per_year.median()),
      ", max =", tracks_per_year.max())

display(tracks_per_year.describe().to_frame().T)
assert X_tracks.shape == (21299, 58), "This notebook reproduces the paper's data snapshot."
assert np.array_equal(years, np.arange(1958, 2023))
assert np.isfinite(X_tracks).all() and ((X_tracks >= 0) & (X_tracks <= 1)).all()
assert len(set(gene_categories)) == 7
tracks_per_year.to_csv(OUTPUT_DIR / "tracks_per_year.csv")

In [ ]:
first_order = np.vstack([X_tracks[track_years == year].mean(axis=0) for year in years])
def ledoit_wolf_correlation(year_matrix: np.ndarray):
    """Estimate a shrinkage covariance matrix and convert it to correlation."""
    estimator = LedoitWolf().fit(year_matrix)
    covariance = estimator.covariance_
    std = np.sqrt(np.clip(np.diag(covariance), 1e-12, None))
    correlation = covariance / np.outer(std, std)
    correlation = np.clip(correlation, -1.0, 1.0)
    np.fill_diagonal(correlation, 1.0)
    return (correlation, float(estimator.shrinkage_))
correlation_matrices = []
shrinkage_values = []
for year in years:
    matrix = X_tracks[track_years == year]
    corr, shrinkage = ledoit_wolf_correlation(matrix)
    correlation_matrices.append(corr)
    shrinkage_values.append(shrinkage)
correlation_matrices = np.asarray(correlation_matrices)
shrinkage_values = np.asarray(shrinkage_values)
upper_triangle = np.triu_indices(len(gene_names), k=1)
second_order = np.asarray([matrix[upper_triangle] for matrix in correlation_matrices])
pair_names = np.array([f'{gene_names[i]} ↔ {gene_names[j]}' for i, j in zip(*upper_triangle)])
print('Second-order shape:', second_order.shape)
print('Unique relationships:', second_order.shape[1])
print('Ledoit–Wolf shrinkage: min = {:.3f}, mean = {:.3f}, max = {:.3f}'.format(shrinkage_values.min(), shrinkage_values.mean(), shrinkage_values.max()))
first_scaler = StandardScaler()
second_scaler = StandardScaler()
first_z = first_scaler.fit_transform(first_order)
second_z = second_scaler.fit_transform(second_order)
p = first_z.shape[1]
q = second_z.shape[1]
print('Standardized first-order block:', first_z.shape)
print('Standardized second-order block:', second_z.shape)
print('Mean variance of first-order features:', np.var(first_z, axis=0).mean())
print('Mean variance of second-order features:', np.var(second_z, axis=0).mean())
def build_joint_representation(first_block_z, second_block_z, lambda_second=1.0):
    p_local = first_block_z.shape[1]
    q_local = second_block_z.shape[1]
    return np.concatenate([first_block_z / np.sqrt(p_local), np.sqrt(lambda_second) * second_block_z / np.sqrt(q_local)], axis=1)
joint = build_joint_representation(first_z, second_z, lambda_second=LAMBDA_SECOND)
first_block_energy = np.mean(np.mean(first_z ** 2, axis=1))
second_block_energy = np.mean(np.mean(second_z ** 2, axis=1))
print('Joint representation shape:', joint.shape)
print('First-order dimensions:', p)
print('Second-order dimensions:', q)
print('Total joint dimensions:', joint.shape[1])
print('Average first-block squared energy:', round(first_block_energy, 6))
print('Average second-block squared energy:', round(second_block_energy, 6))

upper_i, upper_j = upper_triangle
year_track_indices = {int(year): np.where(track_years == year)[0] for year in years}

## 3. Foote novelty and the mean-only baseline

Euclidean distances between joint annual vectors are contrasted by a Gaussian-weighted checkerboard
with a five-year half-window and sigma = 2.5. Peaks must be at least four years apart; six candidates
are retained. The evaluated boundary centers span 1963–2017.

The baseline uses **unstandardized annual means and cosine distance**, with the same Foote detector.
It reproduces an original-style configuration; it is not a controlled ablation isolating correlations,
because coordinate standardization and distance also differ. Curve z-scores are for display and do not
change peak locations or rankings.

In [ ]:
joint_distance = pairwise_distances(joint, metric="euclidean")
def foote_style_novelty_from_distance(distance_matrix, half_window=5, sigma=None):
    """Compute local novelty with a Gaussian-weighted checkerboard kernel."""
    n_times = distance_matrix.shape[0]
    novelty = np.full(n_times, np.nan)
    h = half_window
    size = 2 * h
    checkerboard = np.ones((size, size))
    checkerboard[:h, :h] = -1.0
    checkerboard[h:, h:] = -1.0
    if sigma is None:
        sigma = h / 2
    coordinates = np.arange(size) - (size - 1) / 2
    gaussian_1d = np.exp(-coordinates ** 2 / (2 * sigma ** 2))
    gaussian_2d = np.outer(gaussian_1d, gaussian_1d)
    kernel = checkerboard * gaussian_2d
    denominator = np.sum(np.abs(kernel))
    for center in range(h, n_times - h):
        local_distance = distance_matrix[center - h:center + h, center - h:center + h]
        novelty[center] = np.sum(local_distance * kernel) / denominator
    return novelty
joint_novelty = foote_style_novelty_from_distance(joint_distance, half_window=HALF_WINDOW)
valid_novelty = np.isfinite(joint_novelty)
joint_novelty_z = np.full(len(years), np.nan)
joint_novelty_z[valid_novelty] = (joint_novelty[valid_novelty] - joint_novelty[valid_novelty].mean()) / joint_novelty[valid_novelty].std()
print('Valid novelty years:', valid_novelty.sum())

In [ ]:
def build_blocks_from_index_samples(sampled_indices_by_year):
    first = []
    second = []
    for year in years:
        matrix = X_tracks[sampled_indices_by_year[int(year)]]
        first.append(matrix.mean(axis=0))
        corr, _ = ledoit_wolf_correlation(matrix)
        second.append(corr[upper_i, upper_j])
    return (np.vstack(first), np.vstack(second))

def novelty_zscore(values):
    result = np.full_like(values, np.nan, dtype=float)
    valid_local = np.isfinite(values)
    if valid_local.any():
        mean_local = np.nanmean(values[valid_local])
        std_local = np.nanstd(values[valid_local])
        result[valid_local] = (values[valid_local] - mean_local) / std_local
    return result

def strongest_peak_years(values, top_k=5, minimum_distance=4):
    safe = np.where(np.isfinite(values), values, -np.inf)
    positions, _ = find_peaks(safe, distance=minimum_distance)
    positions = sorted(positions, key=lambda position: values[position], reverse=True)
    return [int(years[position]) for position in positions[:top_k]]
def local_peak_table(values, minimum_distance=4):
    safe_values = np.where(np.isfinite(values), values, -np.inf)
    peak_positions, _ = find_peaks(safe_values, distance=minimum_distance)
    ordered_positions = sorted(peak_positions, key=lambda position: values[position], reverse=True)
    return pd.DataFrame({'rank': np.arange(1, len(ordered_positions) + 1), 'year': years[ordered_positions], 'joint_novelty': values[ordered_positions], 'joint_novelty_z': novelty_zscore(values)[ordered_positions], 'position': ordered_positions})
joint_peaks = local_peak_table(joint_novelty, minimum_distance=MIN_PEAK_DISTANCE)
print('Highest local peaks in the joint novelty curve:')
display(joint_peaks.head(10).drop(columns='position'))
first_order_baseline_distance = pairwise_distances(first_order, metric='cosine')
first_order_baseline_novelty = foote_style_novelty_from_distance(first_order_baseline_distance, half_window=HALF_WINDOW)
baseline_peaks = local_peak_table(first_order_baseline_novelty, minimum_distance=MIN_PEAK_DISTANCE)
print('Highest local peaks from the original-style first-order baseline:')
display(baseline_peaks.head(10)[['rank', 'year', 'joint_novelty']].rename(columns={'joint_novelty': 'first_order_baseline_novelty'}))

## 4. First- and second-order contributions

For boundary t, compare the five annual vectors at t−5,…,t−1 with those at t,…,t+4.
First- and second-order energies are the average squared standardized centroid differences.
Their shares decompose **centroid displacement, not the Foote novelty score**.
Attributes and pairs are ranked within their own blocks by absolute standardized displacement.

In [ ]:
def transition_decomposition(center_index, half_window=5, lambda_second=1.0):
    pre = slice(center_index - half_window, center_index)
    post = slice(center_index, center_index + half_window)
    delta_first_z = first_z[post].mean(axis=0) - first_z[pre].mean(axis=0)
    delta_second_z = second_z[post].mean(axis=0) - second_z[pre].mean(axis=0)
    first_energy = np.mean(delta_first_z ** 2)
    second_energy = lambda_second * np.mean(delta_second_z ** 2)
    total_energy = first_energy + second_energy
    first_share = first_energy / total_energy
    second_share = second_energy / total_energy
    raw_first_change = first_order[post].mean(axis=0) - first_order[pre].mean(axis=0)
    raw_second_change = second_order[post].mean(axis=0) - second_order[pre].mean(axis=0)
    return {'delta_first_z': delta_first_z, 'delta_second_z': delta_second_z, 'first_energy': first_energy, 'second_energy': second_energy, 'first_share': first_share, 'second_share': second_share, 'joint_centroid_distance': np.sqrt(total_energy), 'raw_first_change': raw_first_change, 'raw_second_change': raw_second_change, 'pre_second_raw': second_order[pre].mean(axis=0), 'post_second_raw': second_order[post].mean(axis=0)}
contribution_rows = []
for _, peak in joint_peaks.head(6).iterrows():
    position = int(peak['position'])
    info = transition_decomposition(position, half_window=HALF_WINDOW, lambda_second=LAMBDA_SECOND)
    contribution_rows.append({'rank': int(peak['rank']), 'year': int(peak['year']), 'joint_novelty': peak['joint_novelty'], 'joint_novelty_z': peak['joint_novelty_z'], 'first_order_share': info['first_share'], 'second_order_share': info['second_share'], 'joint_centroid_distance': info['joint_centroid_distance']})
transition_contributions = pd.DataFrame(contribution_rows)
formatted_contributions = transition_contributions.copy()
formatted_contributions['first_order_share'] = (100 * formatted_contributions['first_order_share']).round(1)
formatted_contributions['second_order_share'] = (100 * formatted_contributions['second_order_share']).round(1)
print('Contribution to the squared joint pre/post centroid displacement (%):')
display(formatted_contributions)

In [ ]:
def top_transition_features(year, half_window=5, n_features=5):
    center_index = int(np.where(years == year)[0][0])
    info = transition_decomposition(center_index, half_window=half_window, lambda_second=LAMBDA_SECOND)
    first_idx = np.argsort(np.abs(info['delta_first_z']))[::-1][:n_features]
    second_idx = np.argsort(np.abs(info['delta_second_z']))[::-1][:n_features]
    first_table = pd.DataFrame({'attribute': np.array(gene_names)[first_idx], 'standardized_change': info['delta_first_z'][first_idx], 'raw_mean_change': info['raw_first_change'][first_idx]})
    second_table = pd.DataFrame({'relationship': pair_names[second_idx], 'standardized_change': info['delta_second_z'][second_idx], 'pre_correlation': info['pre_second_raw'][second_idx], 'post_correlation': info['post_second_raw'][second_idx], 'raw_correlation_change': info['raw_second_change'][second_idx]})
    return (first_table, second_table)

first_feature_tables, second_feature_tables = [], []
for year in joint_peaks.head(6)["year"].astype(int):
    first_table, second_table = top_transition_features(year, n_features=10)
    first_feature_tables.append(first_table.assign(year=year))
    second_feature_tables.append(second_table.assign(year=year))
pd.concat(first_feature_tables).to_csv(OUTPUT_DIR / "top_attribute_changes.csv", index=False)
pd.concat(second_feature_tables).to_csv(OUTPUT_DIR / "top_relation_changes.csv", index=False)
display(transition_contributions.round(4))

## 5. Full-pipeline bootstrap

Resample tracks with replacement separately within each year; recompute means, correlations,
both standardizations, joint distances and novelty. Recovery means a peak among the five strongest
within ±1 year of each observed candidate. The default is 5,000 repetitions with the original seed.
These recovery frequencies are stability measures, not probabilities that a historical event occurred.

In [ ]:
rng_boot = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_novelty_z = np.full((BOOTSTRAP_REPS, len(years)), np.nan, dtype=float)
bootstrap_top_years = []
for replicate in range(BOOTSTRAP_REPS):
    if replicate % 250 == 0:
        print(f'Bootstrap: {replicate}/{BOOTSTRAP_REPS}', flush=True)
    sampled = {}
    for year in years:
        original_idx = year_track_indices[int(year)]
        sampled[int(year)] = rng_boot.choice(original_idx, size=len(original_idx), replace=True)
    boot_first, boot_second = build_blocks_from_index_samples(sampled)
    boot_first_z = StandardScaler().fit_transform(boot_first)
    boot_second_z = StandardScaler().fit_transform(boot_second)
    boot_joint = build_joint_representation(boot_first_z, boot_second_z, lambda_second=LAMBDA_SECOND)
    boot_distance = pairwise_distances(boot_joint, metric='euclidean')
    boot_novelty = foote_style_novelty_from_distance(boot_distance, half_window=HALF_WINDOW)
    boot_z = novelty_zscore(boot_novelty)
    bootstrap_novelty_z[replicate] = boot_z
    bootstrap_top_years.append(strongest_peak_years(boot_z, top_k=BOOTSTRAP_TOP_K, minimum_distance=MIN_PEAK_DISTANCE))
observed_candidate_years = joint_peaks.head(6)['year'].astype(int).tolist()
bootstrap_stability_rows = []
for candidate_year in observed_candidate_years:
    matched = [any((abs(bootstrap_year - candidate_year) <= BOOTSTRAP_MATCH_TOLERANCE for bootstrap_year in peak_years)) for peak_years in bootstrap_top_years]
    bootstrap_stability_rows.append({'observed_transition': candidate_year, 'top5_bootstrap_frequency': np.mean(matched), 'matched_replicates': int(np.sum(matched)), 'bootstrap_replicates': BOOTSTRAP_REPS, 'year_tolerance': BOOTSTRAP_MATCH_TOLERANCE})
bootstrap_peak_stability = pd.DataFrame(bootstrap_stability_rows)

bootstrap_peak_stability.to_csv(OUTPUT_DIR / "bootstrap_stability.csv", index=False)
np.savez_compressed(OUTPUT_DIR / "bootstrap_curves.npz", years=years, novelty_z=bootstrap_novelty_z)
display(bootstrap_peak_stability)

## 6. Refit-scale relational null

Independently shuffle each attribute column within each year. This preserves complete annual marginal
distributions and means, but removes the original cross-attribute pairings. Recompute correlations,
refit their across-year standardization, rebuild the joint space and calculate novelty.
The upper-tail Monte Carlo p-value is (1 + number of null scores ≥ observed)/(B + 1), evaluated
**at the same year**. These are pointwise tests, not a family-wise significance test after peak selection.
They do not distinguish changes in genre mixture from changes within genres.

**Random initialization.** `reference/null_rng_start.json` stores the generator state used
to initialize the refit-null simulations. If the state is unavailable or incompatible,
the initialization function reconstructs it by replaying the required shuffle draws.
The file contains no null scores.

In [ ]:
def refit_null_rng():
    counts = [len(year_track_indices[int(year)]) for year in years]
    signature = {"seed": NULL_SEED, "discarded_realizations": 5000,
                 "year_track_counts": counts, "n_attributes": X_tracks.shape[1],
                 "numpy_version": np.__version__}
    state_path = ROOT / "reference" / "null_rng_start.json"
    if state_path.is_file():
        saved = json.loads(state_path.read_text(encoding="utf-8"))
        if saved["signature"] == signature:
            rng = np.random.default_rng(NULL_SEED)
            rng.bit_generator.state = saved["state"]
            return rng
    rng = np.random.default_rng(NULL_SEED)
    # Random draws depend on the shapes, not on attribute values.
    scratch = np.zeros((max(counts), X_tracks.shape[1]), dtype=float)
    for replicate in range(5000):
        if replicate % 1000 == 0:
            print(f"Reconstructing original null stream: {replicate}/5000", flush=True)
        for count in counts:
            for column in range(X_tracks.shape[1]):
                rng.shuffle(scratch[:count, column])
    return rng

def shuffled_second_order_block(rng):
    second = []
    for year in years:
        matrix = X_tracks[year_track_indices[int(year)]].copy()
        for column in range(matrix.shape[1]):
            rng.shuffle(matrix[:, column])
        corr, _ = ledoit_wolf_correlation(matrix)
        second.append(corr[upper_i, upper_j])
    return np.vstack(second)

def null_joint_novelty(second_null):
    second_null_z = StandardScaler().fit_transform(second_null)
    null_joint = build_joint_representation(first_z, second_null_z, lambda_second=LAMBDA_SECOND)
    null_distance = pairwise_distances(null_joint, metric="euclidean")
    return foote_style_novelty_from_distance(null_distance, half_window=HALF_WINDOW)

rng_null = refit_null_rng()
null_refit_curves = np.full((NULL_REFIT_REPS, len(years)), np.nan)
for replicate in range(NULL_REFIT_REPS):
    if replicate % 250 == 0:
        print(f"Refit null: {replicate}/{NULL_REFIT_REPS}", flush=True)
    null_refit_curves[replicate] = null_joint_novelty(shuffled_second_order_block(rng_null))

null_rows = []
for candidate_year in observed_candidate_years:
    position = int(np.where(years == candidate_year)[0][0])
    observed_value = float(joint_novelty[position])
    refit_values = null_refit_curves[:, position]
    exceedances = int(np.sum(refit_values >= observed_value))
    null_rows.append({"year": candidate_year, "observed_joint_novelty": observed_value,
                      "refit_null_mean": refit_values.mean(), "exceedances": exceedances,
                      "refit_null_p": (1 + exceedances) / (NULL_REFIT_REPS + 1),
                      "null_replicates": NULL_REFIT_REPS})
relational_null_summary = pd.DataFrame(null_rows)
relational_null_summary.to_csv(OUTPUT_DIR / "relational_null.csv", index=False)
np.savez_compressed(OUTPUT_DIR / "relational_null_curves.npz", years=years, novelty=null_refit_curves)
display(relational_null_summary)

## 7. Leave-one-category-out validation

Remove one of the seven categories and all relationships involving its attributes, then refit
the entire annual joint pipeline. Compare novelty curves by Spearman correlation and count recovery
of the full model's top five boundaries within ±1 year. This is seven deterministic ablations,
not another Monte Carlo test.

In [ ]:
gene_category_array = np.array(gene_categories)

def build_joint_novelty_for_attribute_subset(attribute_indices):
    attribute_indices = np.asarray(attribute_indices, dtype=int)
    subset_track_matrix = X_tracks[:, attribute_indices]
    n_subset_attributes = len(attribute_indices)
    subset_upper_i, subset_upper_j = np.triu_indices(n_subset_attributes, k=1)
    subset_first = []
    subset_second = []
    for year in years:
        matrix = subset_track_matrix[track_years == year]
        subset_first.append(matrix.mean(axis=0))
        corr, _ = ledoit_wolf_correlation(matrix)
        subset_second.append(corr[subset_upper_i, subset_upper_j])
    subset_first = np.vstack(subset_first)
    subset_second = np.vstack(subset_second)
    subset_first_z = StandardScaler().fit_transform(subset_first)
    subset_second_z = StandardScaler().fit_transform(subset_second)
    subset_joint = build_joint_representation(subset_first_z, subset_second_z, lambda_second=LAMBDA_SECOND)
    subset_distance = pairwise_distances(subset_joint, metric='euclidean')
    subset_novelty = foote_style_novelty_from_distance(subset_distance, half_window=HALF_WINDOW)
    return subset_novelty
leave_one_out_rows = []
for category in sorted(set(gene_categories)):
    retained_indices = np.where(gene_category_array != category)[0]
    loo_novelty = build_joint_novelty_for_attribute_subset(retained_indices)
    valid_pair = np.isfinite(loo_novelty) & np.isfinite(joint_novelty)
    spearman_rho = spearmanr(loo_novelty[valid_pair], joint_novelty[valid_pair]).statistic
    loo_peak_years = strongest_peak_years(loo_novelty, top_k=5, minimum_distance=MIN_PEAK_DISTANCE)
    full_top5 = joint_peaks.head(5)['year'].astype(int).tolist()
    retained_full_top5 = sum((any((abs(loo_year - full_year) <= 1 for loo_year in loo_peak_years)) for full_year in full_top5))
    leave_one_out_rows.append({'removed_category': category, 'remaining_attributes': len(retained_indices), 'novelty_spearman_rho': spearman_rho, 'top5_peak_years': ', '.join(map(str, loo_peak_years)), 'full_top5_transitions_retained': retained_full_top5})
leave_one_category_out = pd.DataFrame(leave_one_out_rows).sort_values('novelty_spearman_rho')

leave_one_category_out.to_csv(OUTPUT_DIR / "leave_one_category_out.csv", index=False)
display(leave_one_category_out.round(4))

## 8. Manuscript Table 1 and reproducibility metadata

The table is computed from the current bootstrap and null-model results.
Full-precision CSVs are exported alongside the rounded manuscript table. Rounded published reference
values are compared only in full mode; differences are displayed rather than overwritten.

In [ ]:
paper_results = (
    transition_contributions.merge(baseline_peaks[["year", "rank"]].rename(columns={"rank": "mean_rank"}), on="year", how="left")
    .merge(bootstrap_peak_stability.rename(columns={"observed_transition": "year"}), on="year")
    .merge(relational_null_summary[["year", "refit_null_p", "exceedances", "null_replicates"]], on="year")
    .sort_values("year").reset_index(drop=True)
)
paper_results.to_csv(OUTPUT_DIR / "paper_results_full_precision.csv", index=False)
paper_table = pd.DataFrame({
    "Year": paper_results.year, "z": paper_results.joint_novelty_z.round(2),
    "Mean rank": paper_results.mean_rank,
    "Boot. (%)": (100 * paper_results.top5_bootstrap_frequency).round(2),
    "Null p": paper_results.refit_null_p.round(4),
    "1st/2nd (%)": [f"{a:.0f}/{b:.0f}" for a, b in zip(100 * paper_results.first_order_share, 100 * paper_results.second_order_share)],
})
display(paper_table)
paper_table.to_csv(OUTPUT_DIR / "table1.csv", index=False)
latex_rows = [f"{int(row['Year'])} & {row['z']:.2f} & {int(row['Mean rank'])} & {row['Boot. (%)']:.2f} & {row['Null p']:.4f} & {row['1st/2nd (%)']} " + chr(92) * 2 for _, row in paper_table.iterrows()]
(OUTPUT_DIR / "table1_rows.tex").write_text("\n".join(latex_rows) + "\n", encoding="utf-8")

if not QUICK_CHECK:
    reference = pd.read_csv(ROOT / "reference" / "published_validation.csv")
    comparison = paper_results.merge(reference, on="year", suffixes=("", "_published"))
    comparison["bootstrap_count_difference"] = comparison.matched_replicates - comparison.matched_replicates_published
    comparison["null_p_matches_4dp"] = comparison.refit_null_p.round(4) == comparison.refit_null_p_published
    display(comparison[["year", "bootstrap_count_difference", "null_p_matches_4dp"]])
    comparison.to_csv(OUTPUT_DIR / "reference_comparison.csv", index=False)

versions = {name: importlib.metadata.version(name) for name in ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "threadpoolctl"]}
metadata = {"mode": "quick_check" if QUICK_CHECK else "paper", "python": platform.python_version(),
            "versions": versions, "bootstrap_reps": BOOTSTRAP_REPS, "null_reps": NULL_REFIT_REPS,
            "bootstrap_seed": BOOTSTRAP_SEED, "null_seed": NULL_SEED,
            "half_window": HALF_WINDOW, "min_peak_distance": MIN_PEAK_DISTANCE,
            "lambda_second": LAMBDA_SECOND,
            "data_sha256": {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in [GENE_METADATA_PATH, GENE_VALUES_PATH]}}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

## 9. Figure 1 — schematic Foote detector

The two regimes below are synthetic, for explanation only. The schematic uses the same centered
Gaussian kernel and novelty function as the data analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
rng = np.random.default_rng(42)
n = 30
boundary = 15
h = 5
before = rng.normal(loc=[0.0, 0.0], scale=0.18, size=(boundary, 2))
after = rng.normal(loc=[2.0, 1.6], scale=0.18, size=(n - boundary, 2))
X = np.vstack([before, after])
D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
D_plot = (D - D.min()) / (D.max() - D.min())
novelty = foote_style_novelty_from_distance(D, half_window=h)
valid = np.isfinite(novelty)
novelty[valid] = (novelty[valid] - novelty[valid].min()) / (novelty[valid].max() - novelty[valid].min())
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.0, 2.45), gridspec_kw={'width_ratios': [1.0, 1.45], 'wspace': 0.32})
im = ax1.imshow(D_plot, origin='lower', cmap='viridis', vmin=0, vmax=1, interpolation='nearest')
ax1.set_title('(a) Distance matrix', fontsize=9)
ax1.set_xlabel('Time', fontsize=8)
ax1.set_ylabel('Time', fontsize=8)
ax1.set_xticks([])
ax1.set_yticks([])
ax1.axvline(boundary - 0.5, linestyle='--', linewidth=0.9, color='white', alpha=0.9)
ax1.axhline(boundary - 0.5, linestyle='--', linewidth=0.9, color='white', alpha=0.9)
window_start = boundary - h - 0.5
rect = Rectangle((window_start, window_start), 2 * h, 2 * h, fill=False, linewidth=1.5, edgecolor='black')
ax1.add_patch(rect)
txtbox = dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.75)
ax1.text(boundary / 2, boundary / 2, 'low distance\n(within)', ha='center', va='center', fontsize=6.6, color='black', bbox=txtbox)
ax1.text(boundary + (n - boundary) / 2, boundary / 2, 'high distance\n(across)', ha='center', va='center', fontsize=6.6, color='black', bbox=txtbox)
ax1.annotate('Foote window', xy=(boundary - 0.5, boundary - 0.5), xycoords='data', xytext=(1.02, 0.8), textcoords='axes fraction', ha='left', va='center', fontsize=7, arrowprops=dict(arrowstyle='->', linewidth=0.9, color='black', shrinkA=2, shrinkB=2))
x = np.arange(n)
ax2.plot(x, novelty, linewidth=1.7)
ax2.axvline(boundary, linestyle='--', linewidth=1)
ax2.set_title('(b) Foote novelty', fontsize=9)
ax2.set_xlabel('Time', fontsize=8)
ax2.set_ylabel('Novelty', fontsize=8)
ax2.set_xlim(h, n - h)
ax2.set_ylim(-0.05, 1.05)
ax2.set_xticks([boundary])
ax2.set_xticklabels(['boundary'], fontsize=7)
ax2.set_yticks([])
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
peak_t = int(np.nanargmax(novelty))
peak_y = novelty[peak_t]
ax2.annotate('novelty peak', xy=(peak_t, peak_y), xytext=(peak_t + 3.0, 0.78), fontsize=7, ha='left', arrowprops=dict(arrowstyle='->', linewidth=0.8))
plt.savefig(FIGURE_DIR / 'foote_schematic.pdf', bbox_inches='tight')
plt.savefig(FIGURE_DIR / 'foote_schematic.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Figure 2 — joint versus mean-only novelty

The five marked years are the validated boundaries discussed in the manuscript; they are fixed
annotations, not a new selection rule inferred from this plotting cell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def zscore_valid_curve(values):
    """Z-standardize only finite novelty values."""
    values = np.asarray(values, dtype=float)
    result = np.full_like(values, np.nan)
    valid = np.isfinite(values)
    result[valid] = (values[valid] - values[valid].mean()) / values[valid].std()
    return result
joint_plot_z = zscore_valid_curve(joint_novelty)
baseline_plot_z = zscore_valid_curve(first_order_baseline_novelty)
robust_years = [1964, 1983, 2002, 2007, 2017]
label_offsets = {1964: (-2.2, 0.55), 1983: (-1.5, 0.65), 2002: (-3.2, 0.75), 2007: (0.8, 0.55), 2017: (-3.0, 0.3)}
fig, ax = plt.subplots(figsize=(3.45, 2.35))
ax.plot(years, baseline_plot_z, linestyle='--', linewidth=1.25, label='Mean-only')
ax.plot(years, joint_plot_z, linestyle='-', linewidth=1.5, label='Joint')
for year in robust_years:
    idx = np.where(years == year)[0]
    if len(idx) == 0:
        continue
    idx = idx[0]
    x = years[idx]
    y = joint_plot_z[idx]
    ax.scatter(x, y, s=16, zorder=3)
    dx, dy = label_offsets[year]
    ax.annotate(str(year), xy=(x, y), xytext=(x + dx, y + dy), fontsize=6.2, ha='left', va='bottom', arrowprops=dict(arrowstyle='->', linewidth=0.6, shrinkA=2, shrinkB=2))
ax.axhline(0, linewidth=0.5, alpha=0.4)
ax.set_xlabel('Transition year', fontsize=7.5)
ax.set_ylabel('Novelty (z)', fontsize=7.5)
ax.set_xticks([1960, 1970, 1980, 1990, 2000, 2010])
ax.tick_params(axis='both', labelsize=6.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=6.5, frameon=False, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2, handlelength=1.4, columnspacing=1.2)
fig.tight_layout(pad=0.4, rect=[0, 0, 1, 0.93])
plt.savefig(FIGURE_DIR / 'novelty_joint_vs_mean.pdf', bbox_inches='tight')
plt.savefig(FIGURE_DIR / 'novelty_joint_vs_mean.png', dpi=600, bbox_inches='tight')
plt.show()

## 11. Figure 3 — attribute-pair interpretation

Build complete pair tables for 2002 and 2007, then plot the ten pairs selected for the manuscript.
These are **illustrative selections**, not an exhaustive or automatically selected top-ten set.
The complete ranking is exported for transparency. Rank 1 is the largest absolute standardized
correlation-coordinate displacement within a boundary.

Left: mean annual regularized correlation before → after the boundary. Right: the larger absolute
shift of the two endpoint means on their original [0,1] scale. Both use five-year windows and average
the annual summaries with equal year weights. The raw values in the two panels are not comparable
effect sizes.

In [ ]:
HALF_WINDOW_INTERPRETATION = HALF_WINDOW
def build_relational_candidate_table(transition_year, half_window=HALF_WINDOW_INTERPRETATION):
    """
    Build one row for each of the 1,653 second-order coordinates
    around a historical boundary.

    Pre window:
        transition_year - 5 ... transition_year - 1

    Post window:
        transition_year ... transition_year + 4
    """
    if transition_year not in years:
        raise ValueError(f'{transition_year} is not present in `years`.')
    center = int(np.where(years == transition_year)[0][0])
    if center - half_window < 0 or center + half_window > len(years):
        raise ValueError('The requested transition does not have enough years on both sides.')
    pre = slice(center - half_window, center)
    post = slice(center, center + half_window)
    pre_years = years[pre]
    post_years = years[post]
    delta_relation_z = second_z[post].mean(axis=0) - second_z[pre].mean(axis=0)
    r_pre = second_order[pre].mean(axis=0)
    r_post = second_order[post].mean(axis=0)
    delta_r = r_post - r_pre
    rank_order = np.argsort(np.abs(delta_relation_z))[::-1]
    ranks = np.empty(len(delta_relation_z), dtype=int)
    ranks[rank_order] = np.arange(1, len(delta_relation_z) + 1)
    mean_pre_raw = first_order[pre].mean(axis=0)
    mean_post_raw = first_order[post].mean(axis=0)
    delta_mean_raw = mean_post_raw - mean_pre_raw
    mean_pre_z = first_z[pre].mean(axis=0)
    mean_post_z = first_z[post].mean(axis=0)
    delta_mean_z = mean_post_z - mean_pre_z
    rows = []
    for pair_idx, (i, j) in enumerate(zip(upper_i, upper_j)):
        attr_1 = gene_names[i]
        attr_2 = gene_names[j]
        cat_1 = gene_categories[i]
        cat_2 = gene_categories[j]
        pre_r = float(r_pre[pair_idx])
        post_r = float(r_post[pair_idx])
        dr = float(delta_r[pair_idx])
        sign_flip = np.sign(pre_r) != np.sign(post_r) and pre_r != 0 and (post_r != 0)
        rows.append({'transition_year': int(transition_year), 'pair_index': int(pair_idx), 'attribute_1': attr_1, 'category_1': cat_1, 'attribute_2': attr_2, 'category_2': cat_2, 'category_pair': f'{cat_1} × {cat_2}', 'relationship': f'{attr_1} ↔ {attr_2}', 'relation_rank_dz': int(ranks[pair_idx]), 'delta_relation_z': float(delta_relation_z[pair_idx]), 'abs_delta_relation_z': float(abs(delta_relation_z[pair_idx])), 'r_pre': pre_r, 'r_post': post_r, 'delta_r': dr, 'abs_delta_r': abs(dr), 'max_abs_r': max(abs(pre_r), abs(post_r)), 'sign_flip': bool(sign_flip), 'attr1_mean_pre': float(mean_pre_raw[i]), 'attr1_mean_post': float(mean_post_raw[i]), 'attr1_delta_mean_raw': float(delta_mean_raw[i]), 'attr1_delta_mean_z': float(delta_mean_z[i]), 'attr2_mean_pre': float(mean_pre_raw[j]), 'attr2_mean_post': float(mean_post_raw[j]), 'attr2_delta_mean_raw': float(delta_mean_raw[j]), 'attr2_delta_mean_z': float(delta_mean_z[j]), 'pre_period': f'{pre_years[0]}-{pre_years[-1]}', 'post_period': f'{post_years[0]}-{post_years[-1]}'})
    df = pd.DataFrame(rows)
    df = df.sort_values('relation_rank_dz').reset_index(drop=True)
    return df

relation_candidates_2002 = build_relational_candidate_table(2002)
relation_candidates_2007 = build_relational_candidate_table(2007)
for year, table in [(2002, relation_candidates_2002), (2007, relation_candidates_2007)]:
    table.to_csv(OUTPUT_DIR / f"relation_candidates_{year}.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
selected_2002 = [('Percussion', 'Explicit Lyrics'), ('Explicit Lyrics', 'Focus on Lyrics'), ('Harmonic Sophistication', 'Electric Sonority'), ('Vocal Register', 'Synthetic Drums'), ('Synthesizer', 'Focus on Form')]
selected_2007 = [('Focus on Rhythmic Groove', 'Focus on Form'), ('Vocal Grittiness', 'Focus on Form'), ('Vocal Grittiness', 'Synthesizer'), ('Backbeat', 'Synthesizer'), ('Social/Political Lyrics', 'Focus on Form')]
short_names = {('Percussion', 'Explicit Lyrics'): 'Percussion – Explicit Lyrics', ('Explicit Lyrics', 'Focus on Lyrics'): 'Explicit Lyrics – Focus on Lyrics', ('Harmonic Sophistication', 'Electric Sonority'): 'Harm. Soph. – Electric Sonority', ('Vocal Register', 'Synthetic Drums'): 'Vocal Register – Synthetic Drums', ('Synthesizer', 'Focus on Form'): 'Synthesizer – Focus on Form', ('Focus on Rhythmic Groove', 'Focus on Form'): 'Rhythmic Groove – Focus on Form', ('Vocal Grittiness', 'Focus on Form'): 'Vocal Grittiness – Focus on Form', ('Vocal Grittiness', 'Synthesizer'): 'Vocal Grittiness – Synthesizer', ('Backbeat', 'Synthesizer'): 'Backbeat – Synthesizer', ('Social/Political Lyrics', 'Focus on Form'): 'Soc./Pol. Lyrics – Focus on Form'}
years_arr = np.asarray(years)
first_order_arr = np.asarray(first_order)
second_order_arr = np.asarray(second_order)
gene_names_list = list(gene_names)
colors = plt.rcParams['axes.prop_cycle'].by_key()['color'][:5]

def year_index(year):
    """Return the annual-array index for a year."""
    idx = np.where(years_arr == year)[0]
    if len(idx) == 0:
        raise ValueError(f'Year {year} not found.')
    return int(idx[0])

def get_relation_row(df, attr1, attr2):
    """Find a pair regardless of attribute ordering."""
    mask = (df['attribute_1'] == attr1) & (df['attribute_2'] == attr2) | (df['attribute_1'] == attr2) & (df['attribute_2'] == attr1)
    result = df.loc[mask]
    if len(result) == 0:
        raise ValueError(f'Relation not found: {attr1} ↔ {attr2}')
    return result.iloc[0]

def prepare_boundary_data(boundary_year, candidate_df, selected_relations):
    """Compute five-year annual-average correlations and endpoint mean shifts."""
    center = year_index(boundary_year)
    pre = slice(center - 5, center)
    post = slice(center, center + 5)
    rows = []
    for attr1, attr2 in selected_relations:
        idx1 = gene_names_list.index(attr1)
        idx2 = gene_names_list.index(attr2)
        relation_row = get_relation_row(candidate_df, attr1, attr2)
        pair_idx = int(relation_row['pair_index'])
        mu1_pre = float(first_order_arr[pre, idx1].mean())
        mu1_post = float(first_order_arr[post, idx1].mean())
        mu2_pre = float(first_order_arr[pre, idx2].mean())
        mu2_post = float(first_order_arr[post, idx2].mean())
        delta_mu1 = mu1_post - mu1_pre
        delta_mu2 = mu2_post - mu2_pre
        max_abs_delta_mu = max(abs(delta_mu1), abs(delta_mu2))
        r_pre = float(second_order_arr[pre, pair_idx].mean())
        r_post = float(second_order_arr[post, pair_idx].mean())
        rows.append({'relationship': short_names[attr1, attr2], 'attr1': attr1, 'attr2': attr2, 'mu1_pre': mu1_pre, 'mu1_post': mu1_post, 'mu2_pre': mu2_pre, 'mu2_post': mu2_post, 'delta_mu1': delta_mu1, 'delta_mu2': delta_mu2, 'max_abs_delta_mu': max_abs_delta_mu, 'r_pre': r_pre, 'r_post': r_post, 'delta_r': r_post - r_pre, 'relation_rank': int(relation_row['relation_rank_dz'])})
    return pd.DataFrame(rows)
data_2002 = prepare_boundary_data(boundary_year=2002, candidate_df=relation_candidates_2002, selected_relations=selected_2002)
data_2007 = prepare_boundary_data(boundary_year=2007, candidate_df=relation_candidates_2007, selected_relations=selected_2007)

def plot_boundary(ax_mean, ax_relation, df, boundary_year, pre_label, post_label):
    """Plot correlations on the left and endpoint mean shifts on the right."""
    n = len(df)
    y = np.arange(n - 1, -1, -1)
    for ax in [ax_mean, ax_relation]:
        ax.set_ylim(-0.5, n - 0.5)
    for i, row_y in enumerate(y):
        if i % 2 == 0:
            color_band = '0.985'
        else:
            color_band = '0.955'
        ax_mean.axhspan(row_y - 0.5, row_y + 0.5, color=color_band, zorder=0)
        ax_relation.axhspan(row_y - 0.5, row_y + 0.5, color=color_band, zorder=0)
    for i, (row_y, (_, row)) in enumerate(zip(y, df.iterrows())):
        color = colors[i]
        value = row['max_abs_delta_mu']
        ax_mean.hlines(y=row_y, xmin=0, xmax=value, color='0.70', linewidth=1.1, zorder=1)
        ax_mean.scatter(value, row_y, s=42, color=color, edgecolor='black', linewidth=0.5, zorder=3)
        ax_mean.annotate(f'{value:.3f}', xy=(value, row_y), xytext=(8, 0), textcoords='offset points', ha='left', va='center', fontsize=6.2, color='0.25', zorder=5)
    ax_mean.set_xlim(0, 0.1)
    ax_mean.set_xticks([0, 0.025, 0.05, 0.075, 0.1])
    ax_mean.set_xticklabels(['0', '.025', '.05', '.075', '.10'])
    ax_mean.set_yticks(y)
    ax_relation.set_yticks(y)
    ax_relation.set_yticklabels(df['relationship'], fontsize=6.7)
    ax_mean.tick_params(axis='y', left=False, labelleft=False)
    ax_mean.tick_params(axis='x', labelsize=6.5)
    ax_relation.tick_params(axis='y', length=0, pad=4)
    ax_mean.grid(axis='x', alpha=0.14, linewidth=0.5)
    ax_mean.set_xlabel('Largest endpoint mean shift\n$\\max(|\\Delta\\mu_1|,|\\Delta\\mu_2|)$', fontsize=7.0)
    ax_relation.axvline(0, color='0.45', linewidth=0.75, zorder=1)
    for i, (row_y, (_, row)) in enumerate(zip(y, df.iterrows())):
        color = colors[i]
        r_pre = row['r_pre']
        r_post = row['r_post']
        ax_relation.annotate('', xy=(r_post, row_y), xytext=(r_pre, row_y), arrowprops=dict(arrowstyle='-|>', mutation_scale=9, linewidth=1.9, color=color, shrinkA=5, shrinkB=5), zorder=2)
        ax_relation.scatter(r_pre, row_y, s=42, facecolor='white', edgecolor=color, linewidth=1.3, zorder=4)
        ax_relation.scatter(r_post, row_y, s=42, facecolor=color, edgecolor=color, linewidth=0.8, zorder=4)
        if abs(r_post - r_pre) < 0.1:
            pre_dx = -0.035
            post_dx = +0.035
        else:
            pre_dx = 0.0
            post_dx = 0.0
        ax_relation.text(r_pre + pre_dx, row_y - 0.18, f'{r_pre:+.2f}', fontsize=6.0, ha='center', va='top', color='0.28', bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=0.15), zorder=5)
        ax_relation.text(r_post + post_dx, row_y + 0.18, f'{r_post:+.2f}', fontsize=6.0, ha='center', va='bottom', color='0.28', bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=0.15), zorder=5)
    ax_relation.set_xlim(-0.55, 0.55)
    ax_relation.set_xticks([-0.5, -0.25, 0, 0.25, 0.5])
    ax_relation.tick_params(axis='x', labelsize=6.5)
    ax_relation.tick_params(axis='y', length=0, labelleft=True, pad=4)
    ax_relation.grid(axis='x', alpha=0.14, linewidth=0.5)
    ax_relation.set_xlabel('Mean annual regularized correlation', fontsize=7.0)
    ax_relation.set_title(f'{pre_label}  →  {post_label}      (○ pre, ● post)', fontsize=7.8, pad=7)
fig = plt.figure(figsize=(7.2, 5.95))
gs = fig.add_gridspec(2, 2, width_ratios=[0.66, 0.34], hspace=0.42, wspace=0.16)
ax_r_2002 = fig.add_subplot(gs[0, 0])
ax_m_2002 = fig.add_subplot(gs[0, 1], sharey=ax_r_2002)
ax_r_2007 = fig.add_subplot(gs[1, 0])
ax_m_2007 = fig.add_subplot(gs[1, 1], sharey=ax_r_2007)
plot_boundary(ax_mean=ax_m_2002, ax_relation=ax_r_2002, df=data_2002, boundary_year=2002, pre_label='1997–2001', post_label='2002–2006')
plot_boundary(ax_mean=ax_m_2007, ax_relation=ax_r_2007, df=data_2007, boundary_year=2007, pre_label='2002–2006', post_label='2007–2011')
fig.subplots_adjust(left=0.33, right=0.98, bottom=0.08, top=0.93)
for ax, title in [(ax_r_2002, 'Relational change'), (ax_m_2002, 'First-order change')]:
    position = ax.get_position()
    fig.text((position.x0 + position.x1) / 2, 0.985, title, fontsize=9.2, fontweight='bold', ha='center', va='top')
for ax, year in [(ax_r_2002, 2002), (ax_r_2007, 2007)]:
    fig.text(0.03, ax.get_position().y1 + 0.015, str(year), fontsize=9.2, fontweight='bold', ha='left', va='bottom')
plt.savefig(FIGURE_DIR / 'relation_driven_exemplars_polished.pdf', bbox_inches='tight')
plt.savefig(FIGURE_DIR / 'relation_driven_exemplars_polished.png', dpi=600, bbox_inches='tight')
plt.show()
final_summary = pd.concat([data_2002.assign(boundary=2002), data_2007.assign(boundary=2007)], ignore_index=True)
display(final_summary.round(4))

final_summary.to_csv(OUTPUT_DIR / "figure3_values.csv", index=False)
assert final_summary.relation_rank.max() / second_order.shape[1] < 0.09
print("Figures saved in", FIGURE_DIR)

## Interpretation limits

- The sample is the chart repertoire, not all music or a rank-weighted measure of public influence.
- Corpus-level correlations can change because genre composition changes, even if each genre is stable.
- Centroid shares describe the chosen standardized, dimension-balanced representation.
- Null p-values are pointwise; bootstrap recovery is lower for 2002 and 2007 than for the three strongest boundaries.
- The 2002 post-window and 2007 pre-window share 2002–2006. These examples describe successive changes.
- The notebook does not claim that raw correlation shifts are larger than raw mean shifts, or that
  the joint method outperforms all covariance-based change-point detectors.